In [8]:
# ✅ Step 1: Imports
from datetime import datetime
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser


import os
from dotenv import load_dotenv

# ✅ Step 2: Load API key
load_dotenv(".env")
google_api_key=os.getenv("GOOGLE_API_KEY")

# 2. Setup LLM and Embeddings
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", temperature=0)

# 3. Load document and split
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
docs = splitter.create_documents([raw_text])

# 4. Define prompts
map_prompt = PromptTemplate.from_template("Summarize:\n\n{context}")
reduce_prompt = PromptTemplate.from_template("Combine the following summaries into a single answer:\n\n{context}")

# 5. Create map + reduce chains
map_chain = map_prompt | llm | StrOutputParser()
reduce_chain = reduce_prompt | llm | StrOutputParser()


# 6. Use RunnableLambda to convert documents
def map_docs(docs):
    return [{"context": doc.page_content} for doc in docs]

def extract_texts(responses):
    return {"context": "\n".join(responses)}

# 7. Combine using RunnableParallel (MapReduce style)
map_reduce_chain = (
    RunnableLambda(map_docs)
    | map_chain.map()
    | RunnableLambda(extract_texts)
    | reduce_chain
)

# 8. Execute
result = map_reduce_chain.invoke(docs)
print("📄 Final Summary:\n", result)


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


📄 Final Summary:
 Created by Harrison Chase, LangChain is a framework designed for building LLM-powered applications—such as chatbots, document Q&A systems, and AI workflows—while supporting advanced features like RAG, agents, memory, and tools.
